# Pitch Quality Model

**Does the shape of a pitch say something about next season that this season's results do not?**

This notebook runs the whole study in Google Colab. Choose **Runtime → Run all**.

The first run downloads six seasons of pitches from Baseball Savant, so it takes a while.
Everything it writes disappears when the Colab session ends, so download the results at the end.

The code lives in the repository, not in this notebook. Open the files in `pitchq/` to read it.

## 1. Get the code

In [ ]:
import os
if not os.path.exists('pitch-quality-model'):
    !git clone -q https://github.com/takizawadaiki/pitch-quality-model.git
%cd pitch-quality-model
!pip -q install -r requirements.txt

## 2. Check the code before trusting it

Fifteen checks run on made-up data where the right answer is known in advance. They should all pass.

In [ ]:
!python -m pytest -q

## 3. Download the pitches

Regular seasons 2021 to 2026. Months already downloaded are skipped, so if this cell is interrupted, run it again.

In [ ]:
!python run.py download

## 4. Train

The model learns from 2021 to 2023 only. 2024 decides when training stops.

In the report below, `r2` on single pitches will be tiny. One pitch is almost all luck; the test that matters is in the next step, where thousands of pitches are averaged per pitcher.

In [ ]:
!python run.py train
import json
report = json.load(open('results/model_report.json'))
print('Training pitches:', f"{report['train_pitches']:,}", '| rounds:', report['rounds'])
for season, v in report['pitch_level'].items():
    print(season, v['role'].ljust(7), 'r2 =', round(v['r2'], 5))
print('Inputs by importance:', list(report['importance_gain'])[:6])

## 5. Judge the ratings against next season

2025 ratings, judged on 2026 results. Read the three tables in this order:

1. Is the rating's link with next season as strong as this season's results?
2. Does adding the rating to this season's results lower the error? Look for an interval that excludes zero.
3. How much of the rating is signal? Compare its split-half figure with actual runs prevented.

In [ ]:
!python run.py evaluate > /dev/null
from IPython.display import Markdown, Image, display
display(Markdown(open('results/summary.md').read()))
display(Image('results/rating_vs_next_season.png', width=640))
display(Image('results/predictors_of_next_season.png', width=640))

## 6. The leaderboard

In [ ]:
import pandas as pd
board = pd.read_csv('results/leaderboard.csv')
display(board.head(15))
display(pd.read_csv('results/pitch_types.csv').head(15))

## 7. Keep the results

This downloads `results.zip`. On GitHub, open the repository, go into the `results` folder, choose **Add file → Upload files**, and drop the unzipped files in.

Or skip this and press **Run workflow** on the repository's Actions tab, which runs the same pipeline and commits the results for you.

In [ ]:
!zip -qr results.zip results -x 'results/model.txt'
from google.colab import files
files.download('results.zip')

## 8. Make it yours

The defaults are a starting point. `RESEARCH_PLAN.md` lists the decisions worth revisiting. Change one thing in `config.py` at a time, judge it on the **development** pair in `results/metrics.json` (2024 to 2025), and write the change in the plan's log before looking at the final pair again.